# Modélisation de topics sur l'échantillon 3

### 1. Nettoyage et préparation de la base

Ce notebook applique le pipeline BERTopic à l'échantillon 3 de l'étude.
Cet échantillon contient les réponses de proches concernant la santé mentale
et les psychothérapies.

Dans cette première partie, nous :
- chargeons le fichier `Sample3_concat.csv`,
- nettoyons la base (colonnes techniques, valeurs manquantes),
- construisons des variables cliniques et contextuelles dérivées :
  - activité professionnelle du répondant et du proche,
  - psychothérapies suivies,
  - diagnostic du proche (catégories cliniques standardisées).

#### 1.1 Fonction de chargement et de nettoyage

Nous définissons une fonction `load_and_clean` qui :
- charge la base brute de l'échantillon 3 ;
- supprime les colonnes purement techniques ;
- regroupe et nettoie :
  - l'activité professionnelle (répondant / proche),
  - les psychothérapies du proche,
  - les diagnostics psychiatriques déclarés pour le proche ;
- crée une colonne `Trouble_Catégorisé` avec des catégories cliniques synthétiques
  (Depressive disorder, Anxiety, Bipolar disorder, etc.).

In [ ]:
import pandas as pd
import numpy as np
import unicodedata

In [ ]:
def load_and_clean(path):
    """
    Charge et nettoie la base de l'échantillon 3.

    Étapes principales :
    - lecture du fichier CSV brut ;
    - suppression des colonnes techniques (date de soumission, page, etc.) ;
    - fusion des colonnes d'activité professionnelle (répondant / proche) en variables lisibles ;
    - regroupement des réponses sur les psychothérapies du proche ;
    - standardisation des noms de psychothérapies ;
    - classification des diagnostics psychiatriques du proche en grandes catégories
      (Depressive disorder, Anxiety, Bipolar disorder, Addiction disorder, etc.) ;
    - création de la variable `Trouble_Catégorisé` à partir du texte libre.

    Paramètres
    ----------
    path : str
        Chemin du fichier CSV brut (ex. 'Data/Sample3_concat.csv').

    Retour
    ------
    df : pd.DataFrame
        Base nettoyée et enrichie, prête pour l'analyse thématique.
    """
    df = pd.read_csv(path, sep = ';')

    # On supprime les colonnes inutiles
    df.drop(columns=['Date de soumission'], inplace=True)
    df.drop(columns=['Dernière page'], inplace=True)
    df.drop(columns = ['Langue de départ'], inplace = True)
    df.drop(columns=["J'accepte"], inplace=True)
    df.drop(columns=["Tête de série"], inplace=True)


    # --- Fusion des informations d'activité pro (répondant / proche) ---
    # (définition de fusion_repondant, fusion_proche, etc.)

    # Colonnes du répondant
    col_repondant_std = "Quelle est votre activité professionnelle ou la dernière activité professionnelle que vous ayez exercé?"
    col_repondant_autre = "Quelle est votre activité professionnelle ou la dernière activité professionnelle que vous ayez exercé? [Autre]"
    
    # Colonnes du proche
    col_proche_std = "Quelle est l'activité professionnelle ou la dernière activité professionnelle que votre proche ait exercé?"
    col_proche_autre = "Quelle est l'activité professionnelle ou la dernière activité professionnelle que votre proche ait exercé? [Autre]"
    
    # Fusion des informations répondant
    def fusion_repondant(row):
        std = str(row[col_repondant_std]).strip() if pd.notna(row[col_repondant_std]) else ""
        autre = str(row[col_repondant_autre]).strip() if pd.notna(row[col_repondant_autre]) else ""
        if std and autre:
            return f"{std}, {autre}"
        return std or autre or "Non renseigné"
    
    # Fusion des informations proche
    def fusion_proche(row):
        std = str(row[col_proche_std]).strip() if pd.notna(row[col_proche_std]) else ""
        autre = str(row[col_proche_autre]).strip() if pd.notna(row[col_proche_autre]) else ""
        if std and autre:
            return f"{std}, {autre}"
        return std or autre or "Non renseigné"
    
    # Création des colonnes fusionnées
    df["Activité professionnelle (répondant)"] = df.apply(fusion_repondant, axis=1)
    df["Activité professionnelle (proche)"] = df.apply(fusion_proche, axis=1)
    
    # Suppression des colonnes d'origine
    df.drop(columns=[
        col_repondant_std, col_repondant_autre,
        col_proche_std, col_proche_autre
    ], inplace=True)






    # Création d'une colonne rassemblant les différentes psychothérapies
    # Sélection automatique des colonnes liées à la question
    colonnes_therapies = [col for col in df.columns if "psychothérapie" in col.lower() and "[autre" not in col.lower()]
    col_suivi = [col for col in df.columns if "Si votre proche a déjà suivi une psychothérapie" in col]
    col_autre = [col for col in df.columns if "psychothérapie" in col.lower() and "[Autre" in col]
    
    # Dictionnaire de mappage : colonne → nom simple à afficher
    libelles_therapies = {
        col: col.split("une")[-1].strip(" ]).") for col in colonnes_therapies if "jamais" not in col.lower()
    }
    
    
    # Création de la colonne fusionnée
    def fusion_therapies(row):
        if any("jamais" in col.lower() and str(row[col]).strip().lower() == "oui" for col in colonnes_therapies):
            return "Aucune psychothérapie"
        
        liste_therapies = [
            libelles_therapies[col]
            for col in libelles_therapies
            if str(row[col]).strip().lower() == "oui"
        ]
        
        return ", ".join(liste_therapies) if liste_therapies else "Non renseigné"
    
    df["Psychothérapies du proche"] = df.apply(fusion_therapies, axis=1)
    
    # Gestion de la colonne [Autre]
    if col_autre:
        df.rename(columns={col_autre[0]: "Psychothérapie - Autre"}, inplace=True)
    
    # Suppression des colonnes d'origine sauf [Autre]
    df.drop(columns=[col for col in colonnes_therapies if col not in col_autre and col not in col_suivi], inplace=True)
    
    
    def standardiser_therapie(texte):
        if not isinstance(texte, str) or not texte.strip():
            return "Non renseigné"
    
        texte = texte.lower()
    
        # Cas directs
        if "aucune" in texte or "jamais" in texte:
            return "Aucune psychothérapie"
        if "psychanalyse" in texte:
            return "Psychanalyse"
        if "tcc" in texte or "cognitive et comportementale" in texte:
            return "TCC"
        if "systemique" in texte or "familiale" in texte:
            return "Thérapie systémique"
        if "humaniste" in texte:
            return "Thérapie humaniste"
        if "soutien" in texte:
            return "Thérapie de soutien"
        if "emdr" in texte:
            return "EMDR"
        if "icv" in texte or "lifespan" in texte:
            return "Intégration du cycle de vie (ICV)"
        if "je ne connais pas" in texte or "don’t know" in texte:
            return "Psychothérapie inconnue"
        if "autre" in texte:
            return "Autre psychothérapie"
    
        # Réponses narratives/floues
        if "qu’est ce qui a été travaillé" in texte or len(texte.split()) > 10:
            return "Psychothérapie non précisée"
    
        return "Autre psychothérapie"
    
    # Application
    df["Psychothérapie standardisée"] = df["Psychothérapies du proche"].apply(standardiser_therapie)
    
    
    # Création d’un tableau normalisé avec les catégories cibles
    categories_tableau = {
        "No psychotherapy": ["aucune psychothérapie", "non renseigné"],
        "Psychoanalysis": ["psychanalyse"],
        "Cognitive behavior therapy": ["tcc", "thérapie cognitive et comportementale"],
        "Systemic therapy": ["thérapie systémique", "familiale"],
        "Humanist therapy": ["thérapie humaniste"],
        "Supportive therapy": ["thérapie de soutien"],
        "Eye Movement Desensitization and Reprocessing (EMDR)": ["emdr"],
        "Lifespan integration": ["icv", "lifespan"],
        "I don’t know the name of my therapy": ["je ne connais pas", "je ne sais pas", "don’t know"],
        "Other": ["autre", "psychothérapie non précisée"]
    }
    
    compteur = {cat: 0 for cat in categories_tableau}
    
    def assigner_categorie(texte):
        if not isinstance(texte, str):
            return
    
        texte = texte.lower()
    
        for categorie, mots_cles in categories_tableau.items():
            if any(mot in texte for mot in mots_cles):
                compteur[categorie] += 1
                return
    
        # Si aucun mot-clé détecté, classé comme "Other"
        compteur["Other"] += 1
    
    # Appliquer au texte brut ou standardisé
    df["Psychothérapie standardisée"].apply(assigner_categorie)
    df.drop(columns=["Psychothérapies du proche", "Psychothérapie - Autre"], inplace=True)



    # Création d'une colonne regroupant les troubles
    # Fonction pour supprimer les accents
    def remove_accents(text):
        return ''.join(
            c for c in unicodedata.normalize('NFD', text)
            if unicodedata.category(c) != 'Mn'
        )
    
    # Fonction de classification
    def classer_troubles(texte):
        if pd.isna(texte):
            return np.nan
    
        # Nettoyage du texte
        texte = texte.lower()
        texte = remove_accents(texte)
    
        categories = set()
    
        if any(mot in texte for mot in ["depression", "depressive", "deprime", "depressif", "ts", "edc"]):
            categories.add("Depression")
    
        if any(mot in texte for mot in ["anxiete", "angoisse", "anxiete generalisee", "stress", "anxieux", "tag"]):
            categories.add("Anxiety")
    
        if any(mot in texte for mot in ["bipolaire","bipolarite", "bi polarité", "bi polaire", "bi-polarité", "pmd"]):
            categories.add("Bipolar disorder")
    
        if any(mot in texte for mot in ["alcool", "alcoolisme", "alcoolique", "addiction", "addictions", "dependance", "drogue", "toxicomanie",
                                        "tabac", "tabagique"
                                       ]):
            categories.add("Addiction disorder")
    
        if any(mot in texte for mot in ["borderline", "personnalite", "manipule", "manipulateur", "trouble de perso"]):
            categories.add("Personality disorder")
    
        if any(mot in texte for mot in ["boulimie", "anorexie", "alimentaire", "alimentation", "tca"]):
            categories.add("Eating disorder")
    
        if any(mot in texte for mot in ["alzheimer", "declin", "memoire", "demence", "cognitif", "desorientation", "confusion",
                                       "parkinson", "ecriture et lecture lentes"]):
            categories.add("Cognitive disorder")
    
        if any(mot in texte for mot in [
            "schizophrenie", "schizophrène", "hallucination", "hallucinations", 
            "delire", "delires", "paranoia", "paranoiaque","paranoïde", "psychose", 
            "psychotique", "trouble psychotique", "dissociation", "dissociatif"
        ]):
            categories.add("Psychotic disorder")
    
        # Cas spéciaux ou flous
        if any(mot in texte for mot in [
            "sspt", "ptsd", "trauma", "toc", "hyperactivite", "insecurite", "trouble", "instabilite", 
            "dyspraxie", "hypersensibilite", "trouble obsessionnel compulsif", "lassitude", "burn out",
            "burnout"
        ]):
            categories.add("Other psychiatric disorder")
    
        if not categories:
            return "Other psychiatric disorder"
        return ", ".join(sorted(categories))
    
    
    # Application sur une colonne de texte libre
    df["Trouble_Catégorisé"] = df["Merci de nous indiquer ici le ou les trouble(s) psychiatriques dont souffre votre proche"].apply(classer_troubles)

    return df


path = "Data/Sample3_concat.csv"
df = load_and_clean(path)

#### 1.2 Chargement de l'échantillon 3

Nous appliquons la fonction de nettoyage à l'échantillon 3 pour obtenir la base
`df` utilisée dans tout le reste du notebook.

### BERTopic (Code plus détaillé dans le notebook de l'Echantillon 2)

In [4]:
def generer_BERTopic_colonnes(
    df, 
    colonnes, 
    langue_modele='distiluse-base-multilingual-cased-v1',
    min_topic_size=5, 
    n_topics=10, 
    n_words_cloud=15
):
    import pandas as pd
    import spacy
    import string
    from bertopic import BERTopic
    from sentence_transformers import SentenceTransformer
    from hdbscan import HDBSCAN
    from umap import UMAP
    from sklearn.feature_extraction.text import CountVectorizer

    nlp = spacy.load("fr_core_news_md")
    resultats_colonnes = {}

    def nettoyer_texte(txt):
        txt = txt.translate(str.maketrans('', '', string.punctuation))
        doc = nlp(txt.lower())
        return ' '.join([token.lemma_ for token in doc 
                         if not token.is_stop and not token.is_punct 
                         and token.lemma_ != '-PRON-' and token.pos_ in ['NOUN', 'ADJ']])

    stopwords_perso = [
        "je", "ne", "pas", "plus", "que", "dans", "de", "des", "les", "pour",
        "c'est", "il", "elle", "ils", "elles", "sur", "avec", "le", "la", "et", "est",
        "en", "au", "aux", "à", "du", "un", "une", "mes", "tes", "ma", "ta", "on",
        'jai', 'nan', 'cest', 'ner', 'quil', 'quils', "faire", "sentir", "mettre", "voir",
        "arriver", "penser", "savoir", "devoir", "pouvoir", "vouloir"
    ]
    vectorizer_model = CountVectorizer(stop_words=stopwords_perso)
    model = SentenceTransformer(langue_modele)

    for col in colonnes:
        print(f"\n🔹 Analyse de la colonne : {col}")
        df_texts = df[col].dropna().astype(str).to_frame(name='Texte_orig')
        df_texts['Texte_clean'] = df_texts['Texte_orig'].apply(nettoyer_texte)
        df_texts = df_texts.drop_duplicates(subset='Texte_clean')
        df_texts = df_texts[df_texts['Texte_clean'].str.split().str.len() >= 3]
        df_texts = df_texts[~df_texts['Texte_clean'].str.fullmatch(r"\d+", na=False)]

        if len(df_texts) < min_topic_size:
            print(f"⚠️ Trop peu de textes valides dans la colonne {col}, sautée.")
            continue

        cleaned_list = df_texts['Texte_clean'].tolist()
        orig_list = df_texts['Texte_orig'].tolist()
        df_texts['Diagnostic'] = df.loc[df_texts.index, 'Trouble_Catégorisé']

        embeddings = model.encode(cleaned_list, show_progress_bar=False)
        umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine')
        hdbscan_model = HDBSCAN(min_cluster_size=2, min_samples=3, metric='euclidean')

        topic_model = BERTopic(
            vectorizer_model=vectorizer_model,
            embedding_model=model,
            hdbscan_model=hdbscan_model,
            umap_model=umap_model,
            min_topic_size=min_topic_size,
            nr_topics=n_topics,
            top_n_words=n_words_cloud,
            language="french"
        )

        topics, probs = topic_model.fit_transform(cleaned_list, embeddings)

        info_topics = topic_model.get_topic_info()
        df_resultats = pd.DataFrame({
            "Texte_orig": orig_list,
            "Texte_clean": cleaned_list,
            "Topic": topics,
            "Probabilité": probs,
            "Diagnostic": df_texts['Diagnostic'].values
        })

        # Scinder les diagnostics multiples
        df_exploded = df_resultats.copy()
        df_exploded['Diagnostic'] = df_exploded['Diagnostic'].str.split(', ')
        df_exploded = df_exploded.explode('Diagnostic')

        total_docs = len(df_exploded)
        info_topics['Prevalence (%)'] = (info_topics['Count'] / total_docs * 100).round(2)

        # Analyse croisée topic / diagnostic (sans -1)
        df_exploded_valid = df_exploded[df_exploded['Topic'] != -1]
        cross_tab = df_exploded_valid.groupby(['Diagnostic', 'Topic']).size().unstack(fill_value=0)
        cross_tab_percent = cross_tab.div(cross_tab.sum(axis=1), axis=0) * 100

        # Transpose pour merge avec le tableau final
        topics_percent = cross_tab_percent.T.reset_index()
        topics_percent.rename(columns={topics_percent.columns[0]: "numéro de topic"}, inplace=True)
        topics_percent["numéro de topic"] = topics_percent["numéro de topic"].astype(int)

        # Résumé des topics
        exemples_par_topic = {}
        for t in info_topics.Topic.unique():
            if t == -1:
                continue
            docs_topic = df_resultats[df_resultats.Topic == t]
            top_exemples = docs_topic.nlargest(2, 'Probabilité')['Texte_orig'].tolist()
            exemples_par_topic[t] = top_exemples

        final_results = []
        for _, row in info_topics.iterrows():
            if row['Topic'] == -1:
                continue
            topic_id = int(row['Topic'])
            best_words = [w for w, _ in topic_model.get_topic(topic_id)[:10]]
            prevalence = row['Prevalence (%)']
            exemples = exemples_par_topic.get(topic_id, ["", ""])
            final_results.append({
                "numéro de topic": topic_id,
                "Name of the topic": ', '.join(best_words),
                "10 best words": ', '.join(best_words),
                "prévalence": prevalence,
                "deux exemples types": " | ".join(exemples),
                "Summary created by the generative IA": ""
            })

        df_final = pd.DataFrame(final_results)
        df_final["numéro de topic"] = df_final["numéro de topic"].astype(int)

        # Fusion diagnostics (les pourcentages de chaque diagnostic par topic)
        df_final = df_final.merge(topics_percent, on="numéro de topic", how="left")

        # Stockage
        resultats_colonnes[col] = {
            "model": topic_model,
            "df_resultats": df_resultats,
            "df_final": df_final
        }

    return resultats_colonnes


### 1.3 Vérification de la base nettoyée

Nous vérifions que la base nettoyée contient bien les colonnes attendues
(questions ouvertes, variables cliniques standardisées).

In [ ]:
# Vérification rapide : aperçu des premières lignes après nettoyage
df.head()

### 1.4 Identification des questions ouvertes à analyser

Nous sélectionnons ici les colonnes correspondant aux questions ouvertes
qui seront analysées avec BERTopic.

- `col_quot` : vécu au quotidien / situations ;
- `col_frein` : freins / obstacles ;
- `col_suivi` : questions liées au suivi en psychothérapie.

Nous concaténons ensuite ces listes dans un objet `col` pour référencer les
questions par index dans la suite du notebook.

In [79]:
df.columns

Index(['ID de la réponse', 'Genre', 'Genre [Autre]', 'Age',
       'Quel est votre niveau d'étude ?', 'Genre de votre proche',
       'Genre de votre proche [Autre]', 'Age de votre proche',
       'Quel est le niveau d'étude de votre proche?', 'Psychothérapie - Autre',
       'Quel lien avait vous avec cette personne?',
       'Quel lien avait vous avec cette personne? [Autre]',
       'Au quotidien, quels sont les comportements (ex : des crises de boulimies) que votre proche a et qui peuvent le faire souffrir, pouvez-vous nous les expliquer ?',
       'Au quotidien, quelles sont les pensées (ex : « Je suis nul(le) ») qui peuvent éventuellement faire souffrir votre proche, pouvez-vous nous les expliquer ?',
       'Au quotidien, quelles sont les émotions (ex : tristesse ») qui peuvent éventuellement faire souffrir votre proche, pouvez-vous nous les expliquer ? ',
       'Au quotidien, quelles sont les relations (ex : relations avec son conjoint(e)) qui peuvent éventuellement faire souf

In [ ]:
# Colonnes contenant les réponses sur le vécu au quotidien / situations
col_quot = [col for col in df.columns if "quotidien" in col or "situations" in col]

# Colonnes contenant les réponses sur les freins / obstacles
col_frein = [col for col in df.columns if "frein" in col]

# Colonnes relatives au suivi psychothérapeutique du proche
col_suivi = [col for col in df.columns if "Si votre proche a déjà suivi une psychothérapie" in col]

col_situations = [col for col in df.columns if "situations" in col]

col_situations

['Quelles sont les situations et/ou les évènements (e.g. situations de violences) qui peuvent éventuellement faire souffrir vos proches, pouvez-vous nous les expliquer\xa0?']

In [ ]:
# Liste unique des colonnes de questions ouvertes utilisées dans ce notebook
col = col_quot + col_frein + col_suivi
col

['Au quotidien, quels sont les comportements (ex\xa0: des crises de boulimies) que votre proche a et qui peuvent le faire souffrir, pouvez-vous nous les expliquer\xa0?',
 'Au quotidien, quelles sont les pensées (ex\xa0: «\xa0Je suis nul(le)\xa0») qui peuvent éventuellement faire souffrir votre proche, pouvez-vous nous les expliquer\xa0?',
 'Au quotidien, quelles sont les émotions (ex\xa0: tristesse\xa0») qui peuvent éventuellement faire souffrir votre proche, pouvez-vous nous les expliquer\xa0? ',
 'Au quotidien, quelles sont les relations (ex\xa0: relations avec son conjoint(e)) qui peuvent éventuellement faire souffrir votre proche, pouvez-vous nous les expliquer\xa0?',
 'Au quotidien, quelles sont les autres choses qui peuvent éventuellement faire souffrir votre proche, pouvez-vous nous les expliquer\xa0?',
 'Que voudriez-vous voir changer dans le quotidien de votre proche\xa0?',
 'Quelles sont les situations et/ou les évènements (e.g. situations de violences) qui peuvent éventuelle

#### 1.5 Correspondance index → question

Pour faciliter la lecture du code, nous utiliserons des index sur la liste `col` :

- `col[0]` : *comportements du proche*  
- `col[1]` : *pensées du proche*
- `col[2]` : *émotions du proche*
- `col[3]` : *relations du proche*  
- `col[4]` : *les autres choses pouvant faire souffrir le proche*
- etc.

Cela permet de réutiliser la même fonction `generer_BERTopic_colonnes` pour plusieurs
questions sans dupliquer le code.

## 2.1 Thématiques : Pensées

Nous appliquons ici BERTopic aux réponses à la question :

> *Au quotidien, quelles sont les pensées (ex : « Je suis nul(le) ») qui peuvent éventuellement faire souffrir votre proche, pouvez-vous nous les expliquer ?*

Paramètres :
- `min_topic_size = 2`
- `n_topics = 12`

In [144]:
# Réglages affichage pandas
pd.set_option('display.max_colwidth', None)

# Sélection des colonnes à afficher, exactement comme l'image 1
info = resultats[col[1]]['info_topics']  # DataFrame attendu

In [10]:
# 1. Lancer l’analyse topic
resultats = generer_BERTopic_colonnes(df, colonnes=[col[1]], langue_modele='distiluse-base-multilingual-cased-v1',
    min_topic_size=2,     # adapter à la taille de tes données
    n_topics=12,        # ou fixe, ex: 3
    n_words_cloud=25      # nombre de mots pour le nuage (pas pour le tableau final)
)

# 2. Extraire le tableau final prêt à l’emploi
df_final = resultats[col[1]]["df_final"]

df_final
df_final.to_excel("topics_etendus_thoughts_E3.xlsx", index=False)





🔹 Analyse de la colonne : Au quotidien, quelles sont les pensées (ex : « Je suis nul(le) ») qui peuvent éventuellement faire souffrir votre proche, pouvez-vous nous les expliquer ?


In [11]:
pd.set_option('display.max_colwidth', None)

print(sum(df_final.prévalence))
df_final

51.44


,numéro de topic,Name of the topic,10 best words,prévalence,deux exemples types,Summary created by the generative IA,Addiction disorder,Anxiety,Bipolar disorder,Cognitive disorder,Depression,Eating disorder,Other psychiatric disorder,Personality disorder,Psychotic disorder
0,0,"pensée, négatif, souffrance, quotidien, type, chose, genre, mal, changement, difficile","pensée, négatif, souffrance, quotidien, type, chose, genre, mal, changement, difficile",15.61,"Mon proche a souvent des pensées très exigeantes et rigides envers lui-même, comme « Ce n’est pas assez bien » ou « Je dois tout contrôler ». Cela le pousse à se surmener, à se culpabiliser et à ne jamais être satisfait, ce qui lui cause une grande souffrance intérieure. | Il exprime souvent des pensées négatives sur ce qui arrive, ce qui va se passer, ce qui s'est passé. ""C'est nul, je m'ennuie, je n'ai jamais rien à faire, on ne m'écoute pas, je ne peux jamais faire ce dont j'ai envie, je n'ai pas envie de faire ce qui est proposé (ex: se lever, s'habiller, manger, sortir, aller à l'école), je dois toujours faire des choses nulles (ce qu'on m'impose de faire).""",,14.285714,32.558140,50.000000,33.333333,29.870130,25.0,34.210526,40.0,36.363636
1,1,"tâche, vie, part, difficulté, fois, chose, cas, inutile, nul, sentiment","tâche, vie, part, difficulté, fois, chose, cas, inutile, nul, sentiment",10.98,"Regrette un investissement qu'il a fait étant jeune, difficulté à passer à l'action alors qu'il veut vendre certains de ses objets | J'imagine ""je suis nulle"" ou un sentiment d'incompréhension, qui la fait sortir de ses gonds",,0.000000,18.604651,22.727273,33.333333,18.181818,0.0,22.368421,20.0,9.090909
2,2,"confiance, manqu, valeur, doute, hauteur, manque, compétence, besoin, gros, bon","confiance, manqu, valeur, doute, hauteur, manque, compétence, besoin, gros, bon",6.36,Sentiment de ne jamais être assez et de ne jamais être à la hauteur pour les parents. | Il pense qu'il n'est pas assez aimé et n'a pas assez confiance en sa qualité de mari et père.,,14.285714,9.302326,9.090909,0.000000,12.987013,0.0,13.157895,30.0,27.272727
3,3,"famille, parent, maison, membre, travail, grand, anorexie, impuissant, décès, incompris","famille, parent, maison, membre, travail, grand, anorexie, impuissant, décès, incompris",4.62,"Je pense qu'elle se sent surtout impuissante et parfois inutile à cause des conséquences de son anorexie, car elle ne peut plus bouger autant qu'avant, conduire, faire des balades, travailler dans le jardin... De plus, elle se sent probablement exclue car elle est incapable de manger autant que les autres et a des restrictions alimentaires importante liées à une pancréatite (qui est en partie la cause de son anorexie). Je pense que ma mère se sent parfois très seule et incomprise. | Je suis plutôt inutile au yeux de la société. Pas de travail, pas de famille",,42.857143,4.651163,0.000000,33.333333,10.389610,25.0,11.842105,0.0,18.181818
4,4,"vie, espoir, sens, sentiment, appartenance, texte, nourriture, jamaid, tannée, gout","vie, espoir, sens, sentiment, appartenance, texte, nourriture, jamaid, tannée, gout",3.18,Elle dit tous les temps elle n'a plus de gout pour la vie | desfois elle me texte et me dise qu'elle est tannée et veux s'enlever la vie,,0.000000,6.976744,4.545455,0.000000,10.389610,50.0,1.315789,0.0,0.000000
5,5,"peur, crise, panique, violence, tunnel, prochain, pleur, accident, age, repetition","peur, crise, panique, violence, tunnel, prochain, pleur, accident, age, repetition",2.60,"peur d'avoir une repetition d'un accident en voiture, pleurs, crises d'anxiete severes. | J'ai peut qu'il m'arrive quelque-chose. J'ai peur de faire une crise de panique.",,0.000000,11.627907,0.000000,0.000000,0.000000,0.0,5.263158,10.0,9.090909
6,6,"fille, monde, soeur, rumination, climat, vie, college, damoureux, conflit, analyse","fille, monde, soeur, rumination, climat, vie, college, damoureux, conflit, analyse",2.31,"""comment me perçois le monde extérieur"" il ce que

## 2.2 Thématiques : Émotions

Nous appliquons ici BERTopic aux réponses à la question :

> *Au quotidien, quelles sont les émotions (ex : tristesse ») qui peuvent éventuellement faire souffrir votre proche, pouvez-vous nous les expliquer ?*

Paramètres :
- `min_topic_size = 1`
- `n_topics = 10`

In [58]:
# Réglages affichage pandas
pd.set_option('display.max_colwidth', None)

# Sélection des colonnes à afficher, exactement comme l'image 1
info = resultats[col[1]]['info_topics']  # DataFrame attendu

df_export = info.copy()
df_export['10 best words'] = df_export['Representation'].apply(extract_first_10_words)

# Pour n'afficher que les colonnes voulues (et éventuellement renommer pour Excel)
df_export = df_export[['Name', '10 best words']]
df_export.rename(columns={"Name": "Name of the topic"}, inplace=True)

df_export

,Name of the topic,10 best words
0,-1_colère_tristesse_frustration_difficile,"colère, tristesse, frustration, difficile, chose, face, émotion, sentiment, vie, quotidien"
1,0_peur_anxiété_tristesse_chose,"peur, anxiété, tristesse, chose, stress, situation, angoisse, colère, émotion, social"
2,1_tristesse_triste_culpabilité_manque,"tristesse, triste, culpabilité, manque, déception, fatigue, colère, vie, vide, quotidien"
3,2_émotion_colère_sentiment_tristesse,"émotion, colère, sentiment, tristesse, soeur, négatif, comportement, chose, an, émotionnel"
4,3_solitude_sentiment_vide_lieu,"solitude, sentiment, vide, lieu, tristesse, monde, ami, manqu, fade, relation"
5,4_stress_limpression_heure_quotidien,"stress, limpression, heure, quotidien, activité, vis, phase, temps, chose, rapport"
6,5_nostalgie_passé_nostalgique_mémoire,"nostalgie, passé, nostalgique, mémoire, année, bon, période, temps, joie, enfant"
7,6_cher_père_famille_ami,"cher, père, famille, ami, meffrayer, décès, dêtre, labsence, entourage, chat"
8,7_avantage_triple_handicap_conséquent,"avantage, triple, handicap, conséquent, mieux, invisible, capacité, difficulté, jalousie, hont"


In [14]:
# 1. Lancer l’analyse topic
resultats = generer_BERTopic_colonnes(df, colonnes=[col[2]], langue_modele='distiluse-base-multilingual-cased-v1',
    min_topic_size=1,     # adapter à la taille de tes données
    n_topics=10,        # ou fixe, ex: 3
    n_words_cloud=25      # nombre de mots pour le nuage (pas pour le tableau final)
)

# 2. Extraire le tableau final prêt à l’emploi
df_final = resultats[col[2]]["df_final"]

df_final
df_final.to_excel("topics_etendus_emotions_E3.xlsx", index=False)




🔹 Analyse de la colonne : Au quotidien, quelles sont les émotions (ex : tristesse ») qui peuvent éventuellement faire souffrir votre proche, pouvez-vous nous les expliquer ? 


In [15]:
pd.set_option('display.max_colwidth', None)

print(sum(df_final.prévalence))
df_final

57.05


,numéro de topic,Name of the topic,10 best words,prévalence,deux exemples types,Summary created by the generative IA,Addiction disorder,Anxiety,Bipolar disorder,Cognitive disorder,Depression,Eating disorder,Other psychiatric disorder,Personality disorder,Psychotic disorder
0,0,"tristesse, colère, manqu, confiance, quotidien, triste, colere, sentiment, frustration, chose","tristesse, colère, manqu, confiance, quotidien, triste, colere, sentiment, frustration, chose",21.33,"Tristesse, anxiété, colère, parfois sentiment de vide | tristesse ou sensation de vide",,42.857143,24.074074,28.571429,14.285714,43.478261,33.333333,31.764706,25.000000,33.333333
1,1,"peur, stress, anxiété, tristesse, angoisse, triste, moment, pensée, émotion, inquiétude","peur, stress, anxiété, tristesse, angoisse, triste, moment, pensée, émotion, inquiétude",15.85,"Anxiété permanente, tristesse, souvenir elle dort pour s’évader de son anxiété ou de ses pensées. | Les émotions qui peuvent la faire souffrir selon les périodes sont la tristesse quand un événement négatif survient, l’anxiété.",,0.000000,46.296296,38.095238,57.142857,19.565217,33.333333,25.882353,16.666667,11.111111
2,2,"colère, culpabilité, face, chat, excessif, situation, rapport, irritabilité, haine, action","colère, culpabilité, face, chat, excessif, situation, rapport, irritabilité, haine, action",5.76,"colère (envers les actions des autres et les situations problématiques auxquelles il fait face et doit résoudre de lui-même), haine (envers sa famille qui priorise le plaisir au travail et aux accomplissements), incompréhension (des actions des autres qui n'agissent pas comme lui aurait fait) | Anxiété et colère. Elle va s'inquiéter facilement lorsque les choses ne se passe pas comme elle le souhaite et prendre le contrôle de la situation. Si d'autres personnes sont en désaccord, elle devient facilement en colère.",,14.285714,3.703704,0.000000,0.000000,7.608696,0.000000,16.470588,25.000000,11.111111
3,3,"solitude, famille, besoin, labsence, propre, limpression, tristesse, personne, moment, soulager","solitude, famille, besoin, labsence, propre, limpression, tristesse, personne, moment, soulager",4.03,"La solitude fait souffrir ma conjointe. J'ai l'impression que lorsqu'elle se sent seule, elle se sent abandonné. | Il se sent triste et isolé par moment. Car il a l'impression que les autre ne vivent pas les mêmes choses que lui et ne le comprennent pas.",,0.000000,5.555556,9.523810,14.285714,7.608696,33.333333,4.705882,0.000000,11.111111
4,4,"émotion, physique, émotionnel, culpabilité, intense, sentiment, larme, présent, difficulté, honte","émotion, physique, émotionnel, culpabilité, intense, sentiment, larme, présent, difficulté, honte",3.17,"je dirais que l'émotion qui peu le faire souffrir est l'amertume et une certaine frustration. Ces émotions sont assez récente car mon conjoint a fait la transition de travail à temps plein dans un emploi qui était bien rémunéré, revalorisant, challengeant etc. Il a donc pris sa retraite et débuté en tant que consultant-expert dans son domaine. Deput quelques mois plusieurs embûches imprévisbles nous ont atteint au point d'avoir à reviser même si nous pouvions garder notre maison... L'amertume des situations à vivre et l'injustice ressentie (nous avons entre autre essuyé une fraude majeure qui nous a fait perdre toutes nos économies) la pilule est difficile à digérer. Ne pas oublier qu'il va sur ses 68 ans cet été, donc l'incertitude de ces évènements le font devenir amer et frustré. | À ma connaissance, mon proche ne présente pas d’émotions récurrentes ou intenses, telles que la tristesse, l’anxiété ou la colère, qui pourraient la faire souffrir au quotidien. Son état émotionnel semble stable et adapté aux situations qu’elle rencontre.",,0.000000,12.962963,9.523810,0.000000,6.521739,0.000000,9.411765,16.666667,11.111111
5,5,"parent, mère, relation, enfant, contact, deuil, bon, vie, colèr, familie","parent, mère, relation, enfant, contact, deuil, bon, vie

## 2.3 Thématiques : Comportements du proche

Nous appliquons ici BERTopic aux réponses à la question :

> *Au quotidien, quels sont les comportements (ex : des crises de boulimies) que votre proche a et qui peuvent le faire souffrir, pouvez-vous nous les expliquer ?*

Paramètres utilisés :
- `min_topic_size = 2` : permet de conserver des topics même peu fréquents
  compte tenu de la taille de l'échantillon 3 ;
- `n_topics = 7` : nombre cible de topics (possibilité de fusion par BERTopic).

L’objectif est d’identifier les grandes thématiques liées aux comportements
observés chez le proche et de décrire leur répartition selon `Trouble_Catégorisé`.

In [16]:
# 1. Lancer l’analyse topic
resultats = generer_BERTopic_colonnes(df, colonnes=[col[0]], langue_modele='distiluse-base-multilingual-cased-v1',
    min_topic_size=2,     # adapter à la taille de tes données
    n_topics=7,        # ou fixe, ex: 3
    n_words_cloud=25      # nombre de mots pour le nuage (pas pour le tableau final)
)

# 2. Extraire le tableau final prêt à l’emploi
df_final = resultats[col[0]]["df_final"]

df_final
df_final.to_excel("topics_etendus_behaviours_E3.xlsx", index=False)



🔹 Analyse de la colonne : Au quotidien, quels sont les comportements (ex : des crises de boulimies) que votre proche a et qui peuvent le faire souffrir, pouvez-vous nous les expliquer ?


#### Vérification de la prévalence des topics

Nous vérifions que la somme des prévalences des topics est proche de 100 %.
Les éventuels écarts peuvent provenir de l'arrondi ou de l'exclusion de certains
documents (bruit, topic -1).

In [17]:
pd.set_option('display.max_colwidth', None)

print(sum(df_final.prévalence))
df_final

50.35


,numéro de topic,Name of the topic,10 best words,prévalence,deux exemples types,Summary created by the generative IA,Addiction disorder,Anxiety,Bipolar disorder,Cognitive disorder,Depression,Eating disorder,Other psychiatric disorder,Personality disorder,Psychotic disorder
0,0,"crise, comportement, quotidien, trouble, mère, panique, peur, temps, chose, maison","crise, comportement, quotidien, trouble, mère, panique, peur, temps, chose, maison",26.68,"Ma mère a beaucoup de craintes par rapport à la mort, qui ont commencées lorsque sa mère (ma grand-mère) est décédée il y a plusieurs années. Elle a aussi vécu une terrible séparation peu de temps après, et je constatais beaucoup de tendances dépressives (isolement, manque d'intérêt, ruminations, etc.). Elle va mieux depuis qu'elle est déménagé dans une maison pour retraitées et qu'elle ne vit plus dans l'appartement qu'elle occupait avec son conjoint. Sinon, sur le plan du TDA, elle peine à focuser son attention sur certaines choses plus de quelques minutes sans être distraite. | Fatigue extrême, crise de panique",,87.5,64.814815,68.965517,20.0,51.086957,20.0,55.681818,54.545455,50.000000
1,1,"motivation, fatigue, activité, manqu, lit, boulimie, envie, journée, temps, semaine","motivation, fatigue, activité, manqu, lit, boulimie, envie, journée, temps, semaine",9.28,"Grosse fatigue, prise de poids | Fatigue qui perdure avec un constant besoin de dormir et une facilité à se fâché.",,0.0,12.962963,20.689655,0.0,26.086957,60.0,13.636364,18.181818,28.571429
2,2,"stress, professionnel, spécial, monde, plupart, facteur, situation, difficulté, sentiment, besoin","stress, professionnel, spécial, monde, plupart, facteur, situation, difficulté, sentiment, besoin",5.57,"Elle fait de l'embonpoint en raison de manger en trop pour rassasier ses facteurs de stress personnels et professionnels. | Probablement son besoin que tout soit exactement comme elle veux, peu importe la situation. La souffrance vient pas d'elle, mais elle vient lorsque la réalité ne conforme pas à ce qu'elle veut.",,0.0,5.555556,0.000000,40.0,5.434783,0.0,18.181818,18.181818,7.142857
3,3,"noir, pensée, dhumeur, saut, triste, boucle, suicidaire, vif, tendance, négatif","noir, pensée, dhumeur, saut, triste, boucle, suicidaire, vif, tendance, négatif",4.18,"Insomnies, pensées suicidaires, idées noires qui tournent en boucle | La plupart du temps, il se sent triste et a des sautes d'humeur. Il reste souvent assis seul, perdu dans ses pensées, et je le trouve souvent triste sur le moment.",,12.5,11.111111,6.896552,0.0,11.956522,20.0,1.136364,9.090909,0.000000
4,4,"perte, mémoire, difficulté, agressivité, conversation, violence, parole, soudain, relation, tâche","perte, mémoire, difficulté, agressivité, conversation, violence, parole, soudain, relation, tâche",3.71,"paroles négatives à l'encontre des autres, excès de colère et violence (ex. frapper des objets, brises des choses) | Ce parent à des difficultés pour conduire sa voirure car perte de mémoire soudaine et confusion donc ne peut plus conduire pour k'instant.",,0.0,3.703704,3.448276,40.0,4.347826,0.0,9.090909,0.000000,14.285714
5,5,"social, ongle, incompréhension, code, niveau, inconnu, agressif, signe, agitation, rumination","social, ongle, incompréhension, code, niveau, inconnu, agressif, signe, agitation, rumination",0.93,"Crise au niveau d’incompréhension sociale, s’énerve de ne pas comprendre, de ne pas avoir les codes pour interpréter | Isolement social, ruminations",,0.0,1.851852,0.000000,0.0,1.086957,0.0,2.272727,0.000000,0.000000


## 2.4 Thématiques : Relations du proche

Nous appliquons ici BERTopic aux réponses à la question :

> *Au quotidien, quelles sont les relations (ex : relations avec son conjoint(e)) qui peuvent éventuellement faire souffrir votre proche, pouvez-vous nous les expliquer ?*

Paramètres :
- `min_topic_size = 2`
- `n_topics = 8`

In [ ]:
# 1. Lancer l’analyse topic
resultats = generer_BERTopic_colonnes(df, colonnes=[col[3]], langue_modele='distiluse-base-multilingual-cased-v1',
    min_topic_size=2,     # adapter à la taille de tes données
    n_topics=8,        # ou fixe, ex: 3
    n_words_cloud=25      # nombre de mots pour le nuage (pas pour le tableau final)
)

# 2. Extraire le tableau final prêt à l’emploi
df_final = resultats[col[3]]["df_final"]

df_final
df_final.to_excel("topics_etendus_relations_E3.xlsx", index=False)


🔹 Analyse de la colonne : Au quotidien, quelles sont les relations (ex : relations avec son conjoint(e)) qui peuvent éventuellement faire souffrir votre proche, pouvez-vous nous les expliquer ?


In [19]:
pd.set_option('display.max_colwidth', None)

print(sum(df_final.prévalence))
df_final

52.92


,numéro de topic,Name of the topic,10 best words,prévalence,deux exemples types,Summary created by the generative IA,Addiction disorder,Anxiety,Bipolar disorder,Cognitive disorder,Depression,Eating disorder,Other psychiatric disorder,Personality disorder,Psychotic disorder
0,0,"relation, mère, enfant, parent, famille, père, conjoint, frère, difficile, problème","relation, mère, enfant, parent, famille, père, conjoint, frère, difficile, problème",26.04,"Ses relations très conflictuelles avec sa famille qui le néglige voire l'insulte, des relations toxiques avec des amis manipulateurs | Relation avec sa mère: la relation a toujours été compliquée, elle veut faire tout pour aider sa fille mais parfois elle est trop invasive et pousse ses choix. elle ne reconnait jamais ses erreurs et préfère éviter les conflits (elle= mère de ma mère)",,71.428571,51.020408,45.0,60.0,59.756098,50.0,47.674419,42.857143,44.444444
1,1,"relation, conjoint, anxiété, proche, peur, cause, couple, source, manque, besoin","relation, conjoint, anxiété, proche, peur, cause, couple, source, manque, besoin",16.34,Relations tendues avec tous proches peuvent bien sûr particulièrement l’affecter. | Il n'a pas de conjoint actuellement. Mais il est souvent timide quand il doit discuter et exprimer ses émotions avec sa partenaire (quand il est en couple).,,14.285714,34.693878,10.0,0.0,23.170732,0.0,32.558140,42.857143,44.444444
2,2,"amie, ami, nouveau, ancien, tête, amical, relation, activite, lont, capable","amie, ami, nouveau, ancien, tête, amical, relation, activite, lont, capable",4.16,"Certaines personnes, collègues, connaissances, peuvent ne pas comprendre le problème et essayer de le ridiculiser . | Des amies qui ne viennent pas la visiter à l’hopital. Des medecins qui annoncent des mauvaises nouvelles.",,0.000000,8.163265,15.0,0.0,6.097561,50.0,9.302326,7.142857,11.111111
3,3,"colère, embarrassant, difficulter, nerveux, minutieux, laccident, semaine, saut, disponible, détail","colère, embarrassant, difficulter, nerveux, minutieux, laccident, semaine, saut, disponible, détail",2.22,"Nous parlons beaucoup ensemble pour essayer de le décharger un peu. Mais lorsque par exemple j'ai eu une grosse semaine et que je ne suis pas disponible pour l'écouter, il essaie de contenir toute cette colère. | il peut être très nerveux et tres minutieux sur chaque détail que ça peut être embarrassant pour les autres",,0.000000,4.081633,10.0,0.0,3.658537,0.0,3.488372,0.000000,0.000000
4,4,"solitude, célibataire, commentaire, lidée, séloigner, rechute, dailleur, colocation, visite, tort","solitude, célibataire, commentaire, lidée, séloigner, rechute, dailleur, colocation, visite, tort",1.66,C'était plus la solitude après avoir était habitué a vivre avec l'autre personne. Reprendre de nouvelles habitudes seuls | Il est célibataire à ce jour et je pense d'ailleurs que c'est une des raisons de la rechute mais peut-être que je me trompe,,0.000000,0.000000,10.0,20.0,2.439024,0.0,3.488372,7.142857,0.000000
5,5,"dispute, lavenir, déclencheur, parole, mensonge, actuel, propos, dialogue, question, dur","dispute, lavenir, déclencheur, parole, mensonge, actuel, propos, dialogue, question, dur",1.39,"Celle avec son conjoint, car c'est conflictuel | On passait des moments ensemble. Nous étions proches mais cela terminé toujours pas des disputes car il tenait des propos déplacés.",,14.285714,2.040816,5.0,0.0,2.439024,0.0,2.325581,0.000000,0.000000
6,6,"femme, maladie, lair, infirmier, enfer, dempathie, ménage, manqu, dernier, mari","femme, maladie, lair, infirmier, enfer, dempathie, ménage, manqu, dernier, mari",1.11,"Un mari qui ne comprends pas la maladie et qui à l'air de ne pas en avoir grand chose a faire ... | il se dispute également avec ses infirmières et femme de ménage, c'est un enfer, il faut en changer régulièrement",,0.000000,0.000000,5.0,20.0,2.439024,0.0,1.162791,0.000000,0.000000


## 2.5 Thématiques : Les Situations pouvant faire souffrir le proche

Nous appliquons ici BERTopic aux réponses à la question :

> *Quelles sont les situations et/ou les évènements (e.g. situations de violences) qui peuvent éventuellement faire souffrir vos proches, pouvez-vous nous les expliquer ?*

Paramètres :
- `min_topic_size = 4`
- `n_topics = 10`

In [22]:
# 1. Lancer l’analyse topic
resultats = generer_BERTopic_colonnes(df, colonnes=col_situations, langue_modele='distiluse-base-multilingual-cased-v1',
    min_topic_size=4,     # adapter à la taille de tes données
    n_topics=10,        # ou fixe, ex: 3
    n_words_cloud=25      # nombre de mots pour le nuage (pas pour le tableau final)
)

# 2. Extraire le tableau final prêt à l’emploi
df_final = resultats[col_situations[0]]["df_final"]

df_final
df_final.to_excel("topics_etendus_situations_E3.xlsx", index=False)


🔹 Analyse de la colonne : Quelles sont les situations et/ou les évènements (e.g. situations de violences) qui peuvent éventuellement faire souffrir vos proches, pouvez-vous nous les expliquer ?


In [24]:
pd.set_option('display.max_colwidth', None)

print(sum(df_final.prévalence))
df_final.drop(columns = ["Cognitive disorder"], inplace = True)
df_final

54.26


,numéro de topic,Name of the topic,10 best words,prévalence,deux exemples types,Summary created by the generative IA,Addiction disorder,Anxiety,Bipolar disorder,Depression,Eating disorder,Other psychiatric disorder,Personality disorder,Psychotic disorder
0,0,"situation, mère, violence, stress, colère, père, chose, verbal, quotidien, conflit","situation, mère, violence, stress, colère, père, chose, verbal, quotidien, conflit",29.79,Ne supporte pas d'entendre mon père s'énerver alors qu'il est colérique. Il le ménage énormément alors qu'il ne devrait pas | Je ne suis pas certaine si je comprends la question. Mais je sais qu'il anticipe beaucoup de façon négative le déclin au niveau cognitif de sa mère et de son père. Il a peur de devenir leur aide-soignant et de les voir vieillir et devenir plus dépendant. Il a aussi peur de sa propre santé.,,33.333333,58.695652,62.962963,58.024691,75.0,56.790123,53.333333,77.777778
1,1,"situation, évènement, evenement, chose, passé, événement, difficile, besoin, question, cause","situation, évènement, evenement, chose, passé, événement, difficile, besoin, question, cause",9.14,Les situations d'injustice ou d'inéquité sont très proprices à faire souffrir ma conjointe. Elle est très sensible à autrui quant à ces dimensions. | Des situations ou il doit aller à des endroits qu'il voudrait à tout prix éviter et il se sent privé de ne pas pouvoir y aller.,,33.333333,23.913043,11.111111,12.345679,0.0,16.049383,20.000000,0.000000
2,2,"violence, dispute, physique, violent, conflit, raison, insulte, verbal, ny, confrontation","violence, dispute, physique, violent, conflit, raison, insulte, verbal, ny, confrontation",7.96,"Comme j'ai écrit plus tôt, je ne sais pas si elle a vécu de la violence, autre que se faire battre pour discipliner, mais ça fait partie de la culture d'Haïti, donc je ne pense pas que ça la fait souffrir tant que ça. | Constater chaque jour la violence du monde",,0.000000,6.521739,18.518519,18.518519,0.0,13.580247,13.333333,11.111111
3,3,"temps, travail, moment, exemple, répond, distance, course, administratif, démarche, emploi","temps, travail, moment, exemple, répond, distance, course, administratif, démarche, emploi",2.36,Le périodes intense/plus charge au travail. | Temps en temps il explose et toute sort en meme temps,,0.000000,2.173913,7.407407,0.000000,0.0,6.172840,0.000000,0.000000
4,4,"famille, ami, dispute, collègue, membre, luimêm, friction, considérationqui, décalge, reunion","famille, ami, dispute, collègue, membre, luimêm, friction, considérationqui, décalge, reunion",1.77,"disputes au quotidien avec famille et personnel aidant | Les disputes qu'il peut avoir avec des membres de sa famille comme il tient beaucoup à eux, des disputes entre amis dont nos disputes, des amis qui ne le prennent pas assez en considération/qui le remplace un peu.",,0.000000,2.173913,0.000000,3.703704,25.0,2.469136,13.333333,0.000000
5,5,"social, inconcevable, effort, réseau, restaurant, renfermement, média, lentreprise, incohérence, secarte","social, inconcevable, effort, réseau, restaurant, renfermement, média, lentreprise, incohérence, secarte",1.18,"Je pense que beaucoup de choses peuvent faire souffrir mes proches. Situation de violence évidemment, mais également de renfermement social, de perte de reconnaissance sociale, plus envie de faire des efforts, cela fait souffrir. | un diner au restaurant est inconcevable pour elle donc elle s'ecarte de tous ces evenements, ce qui est la base de notre cercle social. Donc elle s'isole",,0.000000,2.173913,0.000000,3.703704,0.0,0.000000,0.000000,0.000000
6,6,"fugue, déception, amical, respect, amoureux, amie, fille, manque, petit, relation","fugue, déception, amical, respect, amoureux, amie, fille, manque, petit, relation",0.88,quand sa petite amie a rompu sa relation avec lui et l'a laisse . | ses filles qui lui manque de respect et/ou fugue,,33.333333,2.173913,0.000000,2.469136,0.0,1.234568,0.000000,0.000000
7,7,"amie, fille, décès, lanniversair, bo

## 2.6 Thématiques : les Autres Souffrances du proche

Nous appliquons ici BERTopic aux réponses à la question :

> *Au quotidien, quelles sont les autres choses qui peuvent éventuellement faire souffrir votre proche, pouvez-vous nous les expliquer ?*

Paramètres :
- `min_topic_size = 2`
- `n_topics = 8`

In [25]:
# 1. Lancer l’analyse topic
resultats = generer_BERTopic_colonnes(df, colonnes=[col[4]], langue_modele='distiluse-base-multilingual-cased-v1',
    min_topic_size=2,     # adapter à la taille de tes données
    n_topics=8,        # ou fixe, ex: 3
    n_words_cloud=25      # nombre de mots pour le nuage (pas pour le tableau final)
)

# 2. Extraire le tableau final prêt à l’emploi
df_final = resultats[col[4]]["df_final"]

df_final
df_final.to_excel("topics_etendus_sofferings_E3.xlsx", index=False)


🔹 Analyse de la colonne : Au quotidien, quelles sont les autres choses qui peuvent éventuellement faire souffrir votre proche, pouvez-vous nous les expliquer ?


In [27]:
pd.set_option('display.max_colwidth', None)


print(sum(df_final.prévalence))
df_final.drop(columns = ["Cognitive disorder"], inplace = True)
df_final

49.05


,numéro de topic,Name of the topic,10 best words,prévalence,deux exemples types,Summary created by the generative IA,Addiction disorder,Anxiety,Bipolar disorder,Depression,Eating disorder,Other psychiatric disorder,Personality disorder,Psychotic disorder
0,0,"travail, stress, fatigue, souffrance, difficile, pression, mental, maladie, physique, relation","travail, stress, fatigue, souffrance, difficile, pression, mental, maladie, physique, relation",25.00,"les obligations sociales ou professionnelles, qui amènent de la pression | Ne pas avoir assez de sucre ou de medicaments. Devoir se lever. Devoir se laver (ca va de 1 fois par semaine a 1 fois par mois). Elle a aussi de gros problemes de constipation, pour lesquels elle a du parfois etre hospitalisee, elle passe jusqu'a 2 semaines sans pouvoir aller a la selle.",,33.333333,52.173913,66.666667,59.420290,16.666667,50.724638,61.538462,20.0
1,1,"peur, chose, situation, manque, confiance, rejet, financier, jugement, capable, manière","peur, chose, situation, manque, confiance, rejet, financier, jugement, capable, manière",12.19,"Craintes financières, incapacité à gérer certaines choses administratives par elle-même (je le fais pour elle). Elle n'a pas de petits-enfants, ce qu'elle regrette. Elle est plutôt impulsive, donc parfois a des conflits avec ses amis pour des choses qui me semblent plutôt mineures. Elle réagit émotivement à bien des choses ou des remarques qui seraient pour moi anodines. | Ressentir de l'incompréhension et de l'injustice. La gestion de la frustration est compliquée aussi.",,33.333333,17.391304,11.111111,17.391304,66.666667,28.985507,15.384615,20.0
2,2,"famille, parent, mère, enfant, changement, chose, temps, manque, quotidien, maison","famille, parent, mère, enfant, changement, chose, temps, manque, quotidien, maison",8.75,"Il passe beaucoup trop de temps à jouer à des jeux vidéo, il ne profite pas suffisamment de l'extérieur et passe beaucoup de temps en solitaire. Il n'a pas vraiment de contact avec d'autres gens hormis sa famille. | les relations avec des parents peuvent être difficile",,0.000000,19.565217,11.111111,17.391304,16.666667,17.391304,15.384615,40.0
3,3,"négatif, nouveau, côté, accumulation, anxiogène, actualité, noir, incompris, monde,","négatif, nouveau, côté, accumulation, anxiogène, actualité, noir, incompris, monde,",1.25,Voit du noir et voit que les côtés négatifs | Les actualités (très négatives)\nLe fait de se sentir incompris,,33.333333,6.521739,5.555556,0.000000,0.000000,0.000000,0.000000,0.0
4,4,"fond, sévère, rigide, vulnérabilité, general, jugee, dellemêm, isolation, image, contact","fond, sévère, rigide, vulnérabilité, general, jugee, dellemêm, isolation, image, contact",0.62,"relations sociales en general et isolations, souhaite le contact mais c est parfois difficile, peur d etre jugee | Devoir projeter une image de personne en contrôle et sûre d'elle-même, rigide et sévère dans ses jugements, alors que dans le fond il y a certainement une grande vulnérabilité.",,0.000000,0.000000,0.000000,0.000000,0.000000,1.449275,7.692308,20.0
5,5,"nouveau, intérêt, inattendu, fréquentation, dargent, chose, fois, problème, ,","nouveau, intérêt, inattendu, fréquentation, dargent, chose, fois, problème, ,",0.62,"Certaines fréquentations qui peuvent se rapprocher de lui seulement par intérêt et pour lesquelles il va se plier en quatre juste pour être accepter et ne pas être rejeté une nouvelle fois | Des nouvelles généralement inattendues. des choses auxquelles elle n'était pas préparée et qu'elle ne pouvait pas contrôler, des problèmes d'argent parfois, des choses comme ça.",,0.000000,2.173913,5.555556,2.898551,0.000000,1.449275,0.000000,0.0
6,6,"école, primaire, équipe, professeur, foyer, direction, argent, scolaire, travail, stress","école, primaire, équipe, professeur, foyer, direction, argent, scolaire, travail, stress",0.62,"Le stress de l’école, les travaux scolaire et son travail et l’argent. | Devoir s’occuper seule du foyer et de la 

## 2.7 Thématiques : les Changements espérés chez le proche

Nous appliquons ici BERTopic aux réponses à la question :

> *Que voudriez-vous voir changer dans le quotidien de votre proche ?*

Paramètres :
- `min_topic_size = 2`
- `n_topics = 8`

In [28]:
# 1. Lancer l’analyse topic
resultats = generer_BERTopic_colonnes(df, colonnes=[col[5]], langue_modele='distiluse-base-multilingual-cased-v1',
    min_topic_size=2,     # adapter à la taille de tes données
    n_topics=8,        # ou fixe, ex: 3
    n_words_cloud=25      # nombre de mots pour le nuage (pas pour le tableau final)
)

# 2. Extraire le tableau final prêt à l’emploi
df_final = resultats[col[5]]["df_final"]

df_final
df_final.to_excel("topics_etendus_voeux_E3.xlsx", index=False)


🔹 Analyse de la colonne : Que voudriez-vous voir changer dans le quotidien de votre proche ?


In [29]:
pd.set_option('display.max_colwidth', None)


print(sum(df_final.prévalence))
df_final.drop(columns = ["Cognitive disorder"], inplace = True)
df_final

48.81


,numéro de topic,Name of the topic,10 best words,prévalence,deux exemples types,Summary created by the generative IA,Addiction disorder,Anxiety,Bipolar disorder,Depression,Eating disorder,Other psychiatric disorder,Personality disorder,Psychotic disorder
0,0,"vie, positif, quotidien, stabilité, confiance, petit, goût, chose, heureux, jour","vie, positif, quotidien, stabilité, confiance, petit, goût, chose, heureux, jour",20.36,"J'aimerais que l'on avance dans son diagnostic pour mieux le comprendre et pouvoir l'aider à se sentir mieux, à apprécier ses journées, à voir les choses de manière plus positive, à moins souffrir des changements et des contraintes. | J'aimerais qu'elle soit plus heureuse, même si c'est compliqué. j'aimerais qu'elle continue à peindre et à s'exprimer. j'aimerais pouvoir discuter avec elle pour l'aider mais aussi pour qu'elle m'aide parfois. J'aimerais qu'elle ait des amis.",,25.0,42.5,37.50,45.00,80.0,48.000000,75.000000,30.0
1,1,"mental, activité, crise, sport, santé, physique, pression, stress, symptôme, dactivité","mental, activité, crise, sport, santé, physique, pression, stress, symptôme, dactivité",14.97,Avoir un emploi où il serait reconnu et ne serait pas autant demandant sur les plans physiques et mentaux. | les medicaments qu'il prend sont fort. tesmesta . ils ne font cependant plus tellement d'effet sur lui car son corps est accoutume\n,,50.0,35.0,43.75,33.75,0.0,28.000000,8.333333,40.0
2,2,"indépendant, relation, maison, temps, âge, famille, part, propre, proche, intérêt","indépendant, relation, maison, temps, âge, famille, part, propre, proche, intérêt",5.09,"J'aimerais qu'il prenne plus de temps de lui même. Il se sent coupable lorsqu'il ne passe pas son temps avec sa famille mais des fois il se brûle car il n'accepte pas de l'aide ou n'accepte pas de prendre du temps pour soi (comme du temps calme à la maison ou aller voir des amis, etc). Par contre, ceci s'améliore! | il est dans l'excès des relations amoureuses",,25.0,7.5,6.25,7.50,20.0,6.666667,8.333333,20.0
3,3,"meilleur, travail, prise, société, boulot, rapport, change, organisation, charge, général","meilleur, travail, prise, société, boulot, rapport, change, organisation, charge, général",5.09,"Moins d'auto-critique, meilleure routine, outils pour gérer l'impulsivité, environnement adapté (calme, structuré). | J'aimerais un changement dans sa maniere de penser et enfin se rendre compte qu'il est important qu'elle prenne soin d'elle et faire des choses pout elle meme",,0.0,7.5,6.25,10.00,0.0,8.000000,8.333333,10.0
4,4,"pensée, avilissant, noir, irrationnel, grandchose, fixation, control, mauvais, idée, hygiène","pensée, avilissant, noir, irrationnel, grandchose, fixation, control, mauvais, idée, hygiène",1.50,"Pas grand-chose. Je ne me concerne plus avec ses pensées, pourquoi elle est de telle façon, pourquoi elle fait telle chose, etc. | j'aimerai qu'elle puisse ne plus avoir ces idées noires et se sentiment dévalorisant",,0.0,2.5,0.00,1.25,0.0,5.333333,0.000000,0.0
5,5,"tâche, journée, nuit, long, domestique, couple, voisin, lit, escapade, sommeiléveil","tâche, journée, nuit, long, domestique, couple, voisin, lit, escapade, sommeiléveil",1.20,"Qu’elle se mobilise pour sortir de chez elle\nFaire des tâches domestiques simple\nEntreprenne un rythme sommeil-éveil « normal » dormir la nuit et s’activer le jour. Je voudrais. \n | Plein de chose mais surtout le temps, pouvoir plus me libérer afin de partager des tâches , voire s'offrir une escapade en couple",,0.0,5.0,6.25,2.50,0.0,1.333333,0.000000,0.0
6,6,"total, controle, contrôle, chose, moment, , , , ,","total, controle, contrôle, chose, moment, , , , ,",0.60,Arreter de s inquieter des choses qu elle n a pas entierement le total controle | J'aimerais qu'elle se détende un peu et qu'elle arrête de trop penser à des choses sur lesquelles elle n'a que peu ou pas de contrôle pour le moment.\n,,0.0,0.0,0.00,0.00,0.0,2.666667,0.000000,0.0


## 2.8 Thématiques : Frein à l'amélioration du proche

Nous appliquons ici BERTopic aux réponses à la question :

> *Qu’est-ce qui peut, selon vous, constituer un frein à l’amélioration de l’état de votre proche ?*

Paramètres :
- `min_topic_size = 2`
- `n_topics = 10`

In [30]:
# 1. Lancer l’analyse topic
resultats = generer_BERTopic_colonnes(df, colonnes=[col[7]], langue_modele='distiluse-base-multilingual-cased-v1',
    min_topic_size=2,     # adapter à la taille de tes données
    n_topics=10,        # ou fixe, ex: 3
    n_words_cloud=25      # nombre de mots pour le nuage (pas pour le tableau final)
)

# 2. Extraire le tableau final prêt à l’emploi
df_final = resultats[col[7]]["df_final"]

df_final
df_final.to_excel("topics_etendus_barriers_E3.xlsx", index=False)


🔹 Analyse de la colonne : Qu’est-ce qui peut, selon vous, constituer un frein à l’amélioration de l’état de votre proche ?


In [33]:

print(sum(df_final.prévalence))
#df_final.drop(columns = ["Cognitive disorder"], inplace = True)
df_final

55.81


,numéro de topic,Name of the topic,10 best words,prévalence,deux exemples types,Summary created by the generative IA,Addiction disorder,Anxiety,Bipolar disorder,Depression,Eating disorder,Other psychiatric disorder,Personality disorder,Psychotic disorder
0,0,"frein, amélioration, manque, psychologue, thérapie, peur, travail, environnement, stress, état","frein, amélioration, manque, psychologue, thérapie, peur, travail, environnement, stress, état",23.55,"Le principal frein à l’amélioration de mon proche réside dans sa rigidité mentale et son perfectionnisme excessif, qui l'empêchent de lâcher prise et d’accepter l’imperfection. Il a aussi des difficultés à demander de l’aide, ce qui peut l’isoler, et parfois il manque de confiance dans le processus thérapeutique, ce qui freine ses progrès. | L'etat d'esprit, le surmenage au travail, l'absence de dignostic/traitement de son TDAH",,22.222222,50.000000,43.478261,45.00,0.0,41.772152,33.333333,50.0
1,1,"professionnel, famille, part, comportement, proche, mère, bon, évènement, violence, problème","professionnel, famille, part, comportement, proche, mère, bon, évènement, violence, problème",12.90,"Des évènements choquants. Deuil, violence, etcétéra.. | Le non soutien des proches et des professionnels de santé",,22.222222,10.869565,26.086957,18.75,0.0,24.050633,33.333333,40.0
2,2,"maladie, manque, grand, fatigue, vie, problème, physique, obstacle, caractère, déclic","maladie, manque, grand, fatigue, vie, problème, physique, obstacle, caractère, déclic",10.00,La maladie de quelqu'un proche de lui ou s'il devient malade. | Le fait de ne pas se forcer à faire des choses. c'est difficile mais je pense que ça pourrait être un tremplin. La fatigue et l'état de santé est aussi un frein.,,11.111111,17.391304,21.739130,20.00,100.0,18.987342,33.333333,0.0
3,3,"alcool, drogue, consommation, médicament, local, modérer, monétaire, vérité, lalcool, précontemplation","alcool, drogue, consommation, médicament, local, modérer, monétaire, vérité, lalcool, précontemplation",3.23,"Accès aux drogues et alcohol | le fait qu'il utilise des substances drogues, alcool",,44.444444,4.347826,4.347826,6.25,0.0,3.797468,0.000000,10.0
4,4,"situation, aspect, scénario, panique, mauvbait, imprevue, final, étape, detail, dellemêm","situation, aspect, scénario, panique, mauvbait, imprevue, final, étape, detail, dellemêm",1.61,"elle-même, c'est toujours des situation amplifié et des scénarios qu'elle se fait, mais qui au final tourne toujours bien! | le imprevue et situation ou que lont sait pas les detail ou quelque chose de mauvbait arrive actuaellement",,0.000000,6.521739,0.000000,1.25,0.0,2.531646,0.000000,0.0
5,5,"confiance, manqu, trad, vivide, medecine, image, dinteraction, daccident, jeu, capacité","confiance, manqu, trad, vivide, medecine, image, dinteraction, daccident, jeu, capacité",1.29,Manque de confiance en la medecine trad. | manque de confiance et images vivides d'accidents.,,0.000000,4.347826,0.000000,1.25,0.0,2.531646,0.000000,0.0
6,6,"trou, incompréhension, daide, concret, compréhensive, coincer, attentif, mentalité, accessible, compréhension","trou, incompréhension, daide, concret, compréhensive, coincer, attentif, mentalité, accessible, compréhension",1.29,des personnes pas attentives ou pas compréhensives | Le manque de compréhension et d'aide concrète accessible,,0.000000,4.347826,0.000000,2.50,0.0,1.265823,0.000000,0.0
7,7,"temps, soucier, raison, lénergie, lincapacité, idéal, commentaire, 1an, largent, moment","temps, soucier, raison, lénergie, lincapacité, idéal, commentaire, 1an, largent, moment",1.29,en ce moment je dirais la fac. si elle pouvais se consacrer uniquement a sont amélioration pendant environ 1an sans avoir a se soucier de la fac sa serait idéal pour elle | L'incapacité à accepter de ne pas avoir raison tout le temps,,0.000000,2.173913,0.000000,2.50,0.0,2.531646,0.000000,0.0
8,8,"charge, nonprise, relation, déni, prise, , , , ,","charge, nonprise, relation, déni, prise, , , , ,

## 2.9 Thématiques : Apport de psychothérapie chez le proche

Nous appliquons ici BERTopic aux réponses à la question :

> *Si votre proche a déjà suivi une psychothérapie, qu’est-ce que cela lui a apporté selon vous ?*

Paramètres :
- `min_topic_size = 2`
- `n_topics = 10`

In [34]:
# 1. Lancer l’analyse topic
resultats = generer_BERTopic_colonnes(df, colonnes=[col[8]], langue_modele='distiluse-base-multilingual-cased-v1',
    min_topic_size=2,     # adapter à la taille de tes données
    n_topics=10,        # ou fixe, ex: 3
    n_words_cloud=25      # nombre de mots pour le nuage (pas pour le tableau final)
)

# 2. Extraire le tableau final prêt à l’emploi
df_final = resultats[col[8]]["df_final"]

df_final
df_final.to_excel("topics_etendus_benefits_E3.xlsx", index=False)



🔹 Analyse de la colonne : Si votre proche a déjà suivi une psychothérapie, qu’est-ce que cela lui a apporté selon vous ?


In [35]:
print(sum(df_final.prévalence))
df_final.drop(columns = ["Cognitive disorder"], inplace = True)
df_final

48.78


,numéro de topic,Name of the topic,10 best words,prévalence,deux exemples types,Summary created by the generative IA,Addiction disorder,Anxiety,Bipolar disorder,Depression,Eating disorder,Other psychiatric disorder,Personality disorder,Psychotic disorder
0,0,"thérapie, psychothérapie, relation, émotion, chose, diagnostic, travail, problème, psychologue, suivi","thérapie, psychothérapie, relation, émotion, chose, diagnostic, travail, problème, psychologue, suivi",21.77,"Il a reconnu son problème, mais il lui reste encore à surmonter les difficultés associées à ce diagnostic. Pour l'instant, il n'est pas vraiment en mode solution pour remédier à son problème. | Pour l'instant, nous n'avons fait qu'entamer des bilans. Nous envisageons un suivi en psychothérapie en suivant.",,33.333333,57.142857,37.50,38.888889,100.0,52.631579,62.5,71.428571
1,1,"chose, grand, négatif, vie, heureux, temps, entrain, demdr, dautr, conjoint","chose, grand, négatif, vie, heureux, temps, entrain, demdr, dautr, conjoint",6.85,"pas grand chose parce qu'il n'a pas persevere . et il a toujours refuse de se soigner. | Ils ne parlait pas beaucoup et cetais pas très bon pour lui , plus une perte de temps.",,0.000000,17.857143,18.75,18.518519,0.0,14.035088,12.5,14.285714
2,2,"meilleur, compréhension, situation, connaissance, gestion, important, dellemêm, stabilité, bon, maladie","meilleur, compréhension, situation, connaissance, gestion, important, dellemêm, stabilité, bon, maladie",5.24,"une meilleur comprehension de la situation | une meilleure compréhension de son état, quelques outils pour diminuer un peu le ressenti",,0.000000,3.571429,25.00,9.259259,0.0,8.771930,12.5,14.285714
3,3,"émotion, comportement, point, colère, limpact, contrôle, actuel, repère, réalisation, vue","émotion, comportement, point, colère, limpact, contrôle, actuel, repère, réalisation, vue",5.24,"Réflexion sur ses propres comportements, leur origine et leur façon de fonctionner.\nConfrontation à sa façon de se percevoir et les techniques d'évitement qu'il a mises en place et qu'il faut déconstruire. | Beaucoup plus de calme, de stabilité, de réalisation de son comportement",,0.000000,3.571429,6.25,11.111111,0.0,10.526316,12.5,0.000000
4,4,"soutien, aide, travail, comportement, groupe, dynamique, déchange, déménagement, change, employé","soutien, aide, travail, comportement, groupe, dynamique, déchange, déménagement, change, employé",4.03,"Un support moral, une façon de comprendre pourquoi elle se sentais comme ça, je vois beaucoup d'amélioration depuis. | Un soutien pour traverser cette période et peut-être élaborer une réflexion sur ses engagements au travail, faire de nouveaux choix quant à ses responsabilités.",,33.333333,3.571429,6.25,9.259259,0.0,5.263158,0.0,0.000000
5,5,"stress, crise, anxiété, jeu, esprit, nerveux, mon, lalcoolisme, film, relax","stress, crise, anxiété, jeu, esprit, nerveux, mon, lalcoolisme, film, relax",2.82,"Cela l'a beaucoup aidé pour pouvoir mieux controler son stress quand il arrive, et à pouvoir reprendre les esprits une fois que le stress redescends. | Il a pu être interné, soigné, ne plus avoir de crise. Nous avons pu aussi mettre un mot sur ce qu'il a.",,33.333333,14.285714,0.00,5.555556,0.0,3.508772,0.0,0.000000
6,6,"quelqu, ecoute, avancement, réel, relaxation, écoute, oreille, instant, neutre, outil","quelqu, ecoute, avancement, réel, relaxation, écoute, oreille, instant, neutre, outil",1.21,"Pouvoir s'exprimer, se sentir d'avantage écouter et comprise par une oreille neutre. Pourvoir rendre réel certains mal-être enfouis. | Pour l’instant cela ne lui a pas suffisamment apporté d’avancement. Je pense que cela lui permet de parler et d’avoir quelqu’un qui l’écoute",,0.000000,0.000000,0.00,3.703704,0.0,1.754386,0.0,0.000000
7,7,"langue, accent, maternel, français, embarra, vocabulaire, bipolarité, barrière, manqu, préjugé","langue, accent, maternel, français, embarra, vocabulaire, bipolarité, barrière, manqu, préjugé",0.81,"Pour l’instant, elle a e

## 2.10 Thématiques : Travail de psychothérapie chez le proche

Nous appliquons ici BERTopic aux réponses à la question :

> *Si votre proche a déjà suivi une psychothérapie, qu’est ce qui a été travaillé avec le thérapeute ?*

Paramètres :
- `min_topic_size = 4`
- `n_topics = 10`

In [36]:
# 1. Lancer l’analyse topic
resultats = generer_BERTopic_colonnes(df, colonnes=[col[9]], langue_modele='distiluse-base-multilingual-cased-v1',
    min_topic_size=4,     # adapter à la taille de tes données
    n_topics=10,        # ou fixe, ex: 3
    n_words_cloud=25      # nombre de mots pour le nuage (pas pour le tableau final)
)

# 2. Extraire le tableau final prêt à l’emploi
df_final = resultats[col[9]]["df_final"]

df_final
df_final.to_excel("topics_etendus_targets_E3.xlsx", index=False)



🔹 Analyse de la colonne : Si votre proche a déjà suivi une psychothérapie, qu’est ce qui a été travaillé avec le thérapeute ?


In [37]:
print(sum(df_final.prévalence))
df_final.drop(columns = ["Cognitive disorder"], inplace = True)
df_final

55.29


,numéro de topic,Name of the topic,10 best words,prévalence,deux exemples types,Summary created by the generative IA,Addiction disorder,Anxiety,Bipolar disorder,Depression,Eating disorder,Other psychiatric disorder,Personality disorder,Psychotic disorder
0,0,"relation, rapport, mère, père, grand, parent, familial, famille, enfance, problème","relation, rapport, mère, père, grand, parent, familial, famille, enfance, problème",12.35,"Ses addictions et comportement addictifsn relation avec sa mère, le reste je ne sais pas | Principalement sa relation avec sa mère qui à toujours été très conflictuelle. A travers l'analyse de cette relation, elle à aussi pu cibler plusieurs de ses problèmes d'anxiétés et mieux comprendre ses facons de pensées. Malheureusement je crois que la thérapie s'est terminée trop tôt et aurait bénéficié avoir été plus loin à l'extérieur de sa relation avec sa mère.",,33.333333,34.375,25.000000,21.951220,0.0,13.953488,42.857143,50.0
1,1,"confiance, négatif, pensée, travail, absence, limitante, hygiène, eloignment, bienveillance, dego","confiance, négatif, pensée, travail, absence, limitante, hygiène, eloignment, bienveillance, dego",7.06,"La confiance en soi, sans succès, la créativité | Sa vision très négative de lui même.\nSa quasi absence d'ego et de bienveillance envers lui-même",,0.000000,12.500,0.000000,14.634146,0.0,13.953488,0.000000,0.0
2,2,"session, priver, détail, quiel, type, cbt, quest, qqch, demande, euro","session, priver, détail, quiel, type, cbt, quest, qqch, demande, euro",4.12,"Je ne sais pas. Nous avons décidé de garder le contenu des ses sessions privés, à sa demande. | Oui, plusieurs sessions de CBT, virtuelle pour comprend pourquoi il agit comme il agit.",,0.000000,6.250,8.333333,9.756098,0.0,9.302326,14.285714,0.0
3,3,"trauma, dépression, lenfance, fois, épisode, trouble, douleur, diagnostic, anxiété, noir","trauma, dépression, lenfance, fois, épisode, trouble, douleur, diagnostic, anxiété, noir",11.76,"Il y a eu tentative de travailler sur son anxiété et ses idées noires. | Je ne connais pas tous les détails mais l'aider à gerer sa dépression, les douleurs chroniques, le fait d'avoir frôlée la mort plusieurs fois",,33.333333,21.875,8.333333,24.390244,50.0,25.581395,14.285714,0.0
4,4,"néfaste, mot, stressant, santé, patron, passé, émotion, situation, comportement,","néfaste, mot, stressant, santé, patron, passé, émotion, situation, comportement,",1.18,"le passé, kl'enfance, les situations stressantes, le fait qu'essayer de comprendre les émotions, arriver à mettre des mots desssus | Elle a travaillé sur comprendre ses émotions et comment le gérer et les laisser allez\nElle a travaillé sur comprendre les patrons des comportements et comment déconstruire ceux qui sont néfaste pour sa santé",,0.000000,0.000,0.000000,2.439024,0.0,2.325581,14.285714,0.0
5,5,"situation, quotidien, méditation, productif, reformulation, meilleur, endroit, dimportanc, ancrage, tâche","situation, quotidien, méditation, productif, reformulation, meilleur, endroit, dimportanc, ancrage, tâche",2.35,"Je pense qu'il a travailler sur le niveau d'importance des situations | l'analyse des situations, reformulations des dangers et un meilleur ancrage dans le quotidien + lacher prise",,0.000000,6.250,8.333333,4.878049,0.0,4.651163,0.000000,0.0
6,6,"émotion, gestion, thérapeute, cours, pensée, négatif, travail, compréhension, psychothérapie, place","émotion, gestion, thérapeute, cours, pensée, négatif, travail, compréhension, psychothérapie, place",9.41,"Gestion du temps, techniques anti-procrastination, régulation émotionnelle, communication dans les relations. | Le travail avec le thérapeute a principalement porté sur la gestion de son perfectionnisme et de son besoin de contrôle, ainsi que sur l’amélioration de ses relations interpersonnelles. Il a également exploré comment accepter l’imperfection et réduire certains comportements compulsifs.\n",,0.000000,9.375,41.666667,14.634146,50.0,18.604651,14.285714,50.0
7,7,"tech